# 06 - FD001 current-cycle baselines

Objetivo: entrenar baselines serios para FD001 usando la fila actual de cada ciclo,
con split por motores completos y validacion artificial tipo test.

La evaluacion se hace contra `RUL_raw` sin cap. El target de entrenamiento usa
`RUL` cappeado a 125 ciclos.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessed_FD001 import (
    DEFAULT_CUT_RULS,
    DEFAULT_RUL_CAP,
    prepare_fd001_current_cycle_with_cutoff_eval,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    train_and_predict_models,
)

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = PROJECT_ROOT / "figures" / "fd001"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## Preparacion

Separamos motores completos. La validacion usa cortes artificiales
`cut_ruls = [20, 50, 80, 110, 140]`, no el ultimo ciclo real de los motores
de validacion.

In [2]:
prepared = prepare_fd001_current_cycle_with_cutoff_eval(
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=DEFAULT_CUT_RULS,
    random_state=RANDOM_STATE,
)

display(preprocessing_summary(prepared))
print("Features usadas:", len(prepared["feature_columns"]))
print("Columnas descartadas:", prepared["dropped_columns"])
display(prepared["eval_df"][["unit", "cycle", "cut_rul", "RUL_raw"]].head())

,split,rows,units,features,target_mean,target_min,target_max
0,train,16561,80,17,86.958819,0.0,125.0
1,eval,99,20,17,79.393939,20.0,140.0
2,test_last,100,100,17,75.520000,7.0,145.0


Features usadas: 17
Columnas descartadas: ['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


,unit,cycle,cut_rul,RUL_raw
0,1,172,20,20.0
1,1,142,50,50.0
2,1,112,80,80.0
3,1,82,110,110.0
4,1,52,140,140.0


## Entrenamiento

Modelos comparados:
- `DummyRegressor`
- `Ridge`
- `RandomForestRegressor`
- `MLP` en PyTorch con arquitectura 64-ReLU-Dropout-32-ReLU-1

In [3]:
metrics, validation_predictions, fitted_models = train_and_predict_models(
    prepared,
    representation="current_cycle",
    include_dummy=True,
    random_state=RANDOM_STATE,
)

metrics_path = RESULTS_DIR / "fd001_current_cycle_metrics.csv"
predictions_path = RESULTS_DIR / "fd001_current_cycle_validation_predictions.csv"
bin_metrics_path = RESULTS_DIR / "fd001_current_cycle_metrics_by_rul_bin.csv"

metrics.to_csv(metrics_path, index=False)
validation_predictions.to_csv(predictions_path, index=False)
metrics_by_rul_bin(validation_predictions).to_csv(bin_metrics_path, index=False)

display(metrics)
print("Guardado:", metrics_path)
print("Guardado:", predictions_path)

,representation,model_name,n_features,target_used_for_training,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct
0,current_cycle,RandomForestRegressor,17,RUL_capped,99,16.669382,21.535607,0.739660,1386.500435,14.005055,16.161616
1,current_cycle,MLP,17,RUL_capped,99,16.389131,22.074671,0.726464,1500.262156,15.154163,16.161616
2,current_cycle,Ridge,17,RUL_capped,99,19.620435,24.014162,0.676286,1167.119272,11.789084,24.242424
3,current_cycle,DummyRegressor,17,RUL_capped,99,37.233689,42.879812,-0.032124,18169.638789,183.531705,40.404040


Guardado: C:\Users\frane\udesa\ML\vs\TPF-ML\results\fd001_current_cycle_metrics.csv
Guardado: C:\Users\frane\udesa\ML\vs\TPF-ML\results\fd001_current_cycle_validation_predictions.csv


## Graficos de diagnostico

Los graficos se guardan en `figures/fd001/`.

In [4]:
plot_validation_diagnostics(
    validation_predictions,
    figure_dir=FIGURE_DIR,
    prefix="FD001 current-cycle",
)

sorted(FIGURE_DIR.glob("*.png"))

[WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001/error_vs_true_rul.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001/mae_by_rul_bin.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001/predicted_vs_true.png'),
 WindowsPath('C:/Users/frane/udesa/ML/vs/TPF-ML/figures/fd001/worst_cases_abs_error.png')]

## Lectura rapida

Esta validacion simula el escenario real: para cada motor solo vemos hasta un
corte previo a la falla y medimos contra el RUL restante verdadero sin cap.